# Canadian Housing Affordability Analysis

My generation is being priced out of Canadian cities. This project quantifies exactly how that happened. 
- Which markets moved fastest? 
- When affordability broke down? 
- How much worse things got after COVID? 

Using real Statistics Canada data, it combines new housing prices with median household income to show not just that prices rose, but that they outpaced what people actually earn.

**Data:** Statistics Canada 
- Table 18-10-0205-01 (New Housing Price Index)
- Table 11-10-0190-01 (Median After-Tax Income)

**Cities:** Toronto, Vancouver, Calgary, Edmonton, Montreal, Ottawa, Winnipeg

**Stack:** Python, SQLite, SQL, Plotly, Gemini API

---
## 1. Setup

Connect to the SQLite database and load the libraries used throughout the analysis.

In [1]:
import os
import sqlite3

from google import genai
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

# Connect to housing.db (same directory as this notebook)
DB_PATH = os.path.join(os.path.dirname(os.path.abspath("housing_analysis.ipynb")), "housing.db")
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row   # named-column access if needed

print(f"✓ Connected to:  {DB_PATH}")
print(f"  SQLite version: {sqlite3.sqlite_version}")


✓ Connected to:  /Users/clementho/Desktop/housing-analysis/housing.db
  SQLite version: 3.49.1


## 2. Data Exploration

A quick sanity check before the analysis. We verify the tables loaded correctly, confirm year coverage, and look at a sample of raw rows from each table.

In [2]:
# Database objects
objects = pd.read_sql_query(
    "SELECT type, name FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name",
    conn,
)
print("Database objects")
display(objects)

# Row counts and year coverage
print()
for obj in ("housing_price_index", "median_income", "affordability"):
    row = pd.read_sql_query(
        f"SELECT COUNT(*) AS rows, MIN(year) AS min_year, MAX(year) AS max_year FROM {obj}",
        conn,
    ).iloc[0]
    print(f"  {obj:<25}  {int(row.rows):>4} rows   years {int(row.min_year)}–{int(row.max_year)}")

# Sample rows
print("\nhousing_price_index (first 14 rows)")
display(pd.read_sql_query(
    "SELECT * FROM housing_price_index ORDER BY city, year LIMIT 14", conn
))

print("\nmedian_income (first 14 rows)")
display(pd.read_sql_query(
    "SELECT * FROM median_income ORDER BY city, year LIMIT 14", conn
))

print("\naffordability view (first 14 rows)")
display(pd.read_sql_query(
    "SELECT * FROM affordability ORDER BY city, year LIMIT 14", conn
))


Database objects


,type,name
0,table,housing_price_index
1,table,median_income
2,view,affordability



  housing_price_index         189 rows   years 2000–2026
  median_income               168 rows   years 2000–2023
  affordability               168 rows   years 2000–2023

housing_price_index (first 14 rows)


,year,city,price_index
0,2000,Calgary,42.7000
1,2001,Calgary,43.7417
2,2002,Calgary,46.0417
3,2003,Calgary,48.4500
4,2004,Calgary,51.1333
5,2005,Calgary,54.6833
6,2006,Calgary,78.5583
7,2007,Calgary,91.2417
8,2008,Calgary,91.8333
9,2009,Calgary,85.7083



median_income (first 14 rows)


,year,city,median_income
0,2000,Calgary,67600.0
1,2001,Calgary,73700.0
2,2002,Calgary,74100.0
3,2003,Calgary,66600.0
4,2004,Calgary,74400.0
5,2005,Calgary,72100.0
6,2006,Calgary,79700.0
7,2007,Calgary,83400.0
8,2008,Calgary,82200.0
9,2009,Calgary,84500.0



affordability view (first 14 rows)


,year,city,price_index,median_income,affordability_index,yoy_price_change
0,2000,Calgary,42.7000,67600.0,42.7000,NaN
1,2001,Calgary,43.7417,73700.0,40.1213,2.44
2,2002,Calgary,46.0417,74100.0,42.0030,5.26
3,2003,Calgary,48.4500,66600.0,49.1775,5.23
4,2004,Calgary,51.1333,74400.0,46.4598,5.54
5,2005,Calgary,54.6833,72100.0,51.2703,6.94
6,2006,Calgary,78.5583,79700.0,66.6316,43.66
7,2007,Calgary,91.2417,83400.0,73.9561,16.15
8,2008,Calgary,91.8333,82200.0,75.5223,0.65
9,2009,Calgary,85.7083,84500.0,68.5666,-6.67


## 3. SQL Analysis

Four questions that drive the analysis. The SQL is written out in full so the logic is transparent.

### Which cities are least affordable?

Cities ranked by average affordability index since 2010. The index measures price growth relative to income growth, using 2000 as the baseline. A value above 1.0 means house prices have risen faster than incomes. Higher is worse.

In [3]:
q1_sql = """
SELECT
    a.city,
    ROUND(AVG(a.affordability_index), 2)   AS avg_afford_since_2010,
    ROUND(MAX(a.affordability_index), 2)   AS peak_afford_index,
    p.year                                 AS peak_year,
    ROUND(a2.affordability_index, 2)       AS latest_afford_index
FROM affordability a
JOIN (
    -- year in which each city hit its peak affordability index
    SELECT city, year
    FROM affordability
    WHERE (city, affordability_index) IN (
        SELECT city, MAX(affordability_index)
        FROM affordability
        GROUP BY city
    )
) p ON a.city = p.city
JOIN (
    -- latest available year's value
    SELECT city, affordability_index
    FROM affordability
    WHERE year = (SELECT MAX(year) FROM affordability)
) a2 ON a.city = a2.city
WHERE a.year >= 2010
GROUP BY a.city
ORDER BY avg_afford_since_2010 DESC
"""

df_q1 = pd.read_sql_query(q1_sql, conn)
display(df_q1)


,city,avg_afford_since_2010,peak_afford_index,peak_year,latest_afford_index
0,Ottawa,97.51,142.93,2022,138.49
1,Toronto,92.58,105.46,2022,103.23
2,Montreal,88.49,112.60,2022,112.53
3,Winnipeg,85.88,115.20,2022,111.65
4,Vancouver,83.12,97.66,2022,97.31
5,Calgary,76.97,96.27,2023,96.27
6,Edmonton,76.92,86.66,2007,82.91


### How have prices changed year over year in the four largest markets?

Looking at Toronto, Vancouver, Calgary, and Montreal to see whether price growth has been gradual or driven by sudden spikes. Steady growth and boom-bust cycles tell very different stories about a market.

In [4]:
q2_sql = """
SELECT
    year,
    city,
    ROUND(price_index, 2)       AS price_index,
    ROUND(yoy_price_change, 2)  AS yoy_pct_change
FROM affordability
WHERE city IN ('Toronto', 'Vancouver', 'Calgary', 'Montreal')
  AND year >= 2005
ORDER BY year, city
"""

df_q2 = pd.read_sql_query(q2_sql, conn)

# Pivot for a compact view
df_q2_pivot = df_q2.pivot(index="year", columns="city", values="yoy_pct_change").round(2)
df_q2_pivot.columns.name = "YoY % Change"
print("Year-over-year price change (%) — Toronto, Vancouver, Calgary, Montreal")
display(df_q2_pivot)


Year-over-year price change (%) — Toronto, Vancouver, Calgary, Montreal


YoY % Change,Calgary,Montreal,Toronto,Vancouver
year,,,,
2005,6.94,5.02,4.51,4.38
2006,43.66,4.25,3.82,6.86
2007,16.15,4.28,2.69,7.14
2008,0.65,4.90,3.55,2.29
2009,-6.67,2.35,-0.16,-6.31
2010,1.78,3.11,2.58,3.27
2011,-0.15,2.92,4.66,-0.31
2012,1.67,1.37,5.13,-0.54
2013,5.36,0.97,2.46,-1.03


### Which city appreciated the fastest since 2010?

Total price growth from 2010 to 2023 per city. The simplest measure of how much more expensive each market became over 13 years.

In [5]:
q3_sql = """
SELECT
    b.city,
    ROUND(b.price_index, 2)                                            AS index_2010,
    ROUND(l.price_index, 2)                                            AS index_2023,
    ROUND((l.price_index - b.price_index) * 100.0 / b.price_index, 1) AS total_appreciation_pct
FROM
    (SELECT city, price_index FROM housing_price_index WHERE year = 2010) b
    JOIN (SELECT city, price_index FROM housing_price_index WHERE year = 2023) l
        ON b.city = l.city
ORDER BY total_appreciation_pct DESC
"""

df_q3 = pd.read_sql_query(q3_sql, conn)
display(df_q3)

winner = df_q3.iloc[0]
print(f"\nFastest appreciation: {winner.city} at +{winner.total_appreciation_pct:.1f}% since 2010")


,city,index_2010,index_2023,total_appreciation_pct
0,Ottawa,95.32,170.77,79.2
1,Winnipeg,82.46,141.08,71.1
2,Montreal,92.96,151.64,63.1
3,Toronto,76.48,115.06,50.4
4,Calgary,87.23,126.47,45.0
5,Vancouver,96.57,128.23,32.8
6,Edmonton,97.72,107.52,10.0



Fastest appreciation: Ottawa at +79.2% since 2010


### How bad was the COVID price spike?

Between 2020 and 2022, low interest rates, remote work, and pent-up demand collided. We isolate the 2021 year-over-year jump for each city and show the full 2019 to 2022 window for context.

In [6]:
# 2021 YoY spike ranked
q4a_sql = """
SELECT
    city,
    ROUND(yoy_price_change, 2) AS yoy_2021_pct,
    ROUND(price_index, 2)      AS price_index_2021
FROM affordability
WHERE year = 2021
ORDER BY yoy_2021_pct DESC
"""

print("2021 Year-over-Year Price Spike (ranked)")
df_q4a = pd.read_sql_query(q4a_sql, conn)
display(df_q4a)

# Context: 2019–2022 across all cities
q4b_sql = """
SELECT
    year,
    city,
    ROUND(price_index, 2)      AS price_index,
    ROUND(yoy_price_change, 2) AS yoy_pct_change
FROM affordability
WHERE year BETWEEN 2019 AND 2022
ORDER BY city, year
"""

print("\nContext: price index and YoY change 2019–2022")
df_q4b = pd.read_sql_query(q4b_sql, conn)
df_q4b_pivot = df_q4b.pivot(index="year", columns="city", values="yoy_pct_change").round(2)
df_q4b_pivot.columns.name = "YoY % Change"
display(df_q4b_pivot)


2021 Year-over-Year Price Spike (ranked)


,city,yoy_2021_pct,price_index_2021
0,Ottawa,22.46,154.93
1,Montreal,17.01,135.53
2,Winnipeg,14.79,122.62
3,Vancouver,10.92,120.72
4,Calgary,8.95,105.56
5,Toronto,6.69,110.42
6,Edmonton,4.57,102.10



Context: price index and YoY change 2019–2022


YoY % Change,Calgary,Edmonton,Montreal,Ottawa,Toronto,Vancouver,Winnipeg
year,,,,,,,
2019,-1.71,-1.27,3.88,5.45,-0.88,-1.38,0.80
2020,-1.07,-0.65,7.87,11.55,0.92,1.33,2.01
2021,8.95,4.57,17.01,22.46,6.69,10.92,14.79
2022,16.33,7.15,10.99,11.68,4.52,6.47,15.58


## 4. Visualizations

Four interactive charts. Hover over any point to compare cities at the same year.

### Price index over time, all cities

The raw price index for new housing across all seven cities. The index is set to 100 in December 2016, so a value of 150 means prices are 50% higher than that month. Watch for when the lines start diverging.

In [7]:
df_hpi = pd.read_sql_query(
    "SELECT year, city, price_index FROM housing_price_index ORDER BY city, year",
    conn,
)

fig1 = px.line(
    df_hpi,
    x="year",
    y="price_index",
    color="city",
    markers=True,
    title="New Housing Price Index — All Cities (2016 = 100)",
    labels={"year": "Year", "price_index": "Price Index", "city": "City"},
)
fig1.update_traces(marker_size=4)
fig1.update_layout(
    hovermode="x unified",
    template="plotly_white",
    legend_title_text="City",
    yaxis_title="Price Index (2016 = 100)",
)
fig1.show()


### Total price appreciation 2010 to 2023

A single number per city: how much did new home prices grow over 13 years? This cuts through year-to-year noise and shows the full picture.

In [8]:
df_appr = pd.read_sql_query(
    """
    SELECT
        b.city,
        ROUND((l.price_index - b.price_index) * 100.0 / b.price_index, 1) AS appreciation_pct
    FROM
        (SELECT city, price_index FROM housing_price_index WHERE year = 2010) b
        JOIN (SELECT city, price_index FROM housing_price_index WHERE year = 2023) l
            ON b.city = l.city
    ORDER BY appreciation_pct DESC
    """,
    conn,
)

fig2 = px.bar(
    df_appr,
    x="city",
    y="appreciation_pct",
    color="appreciation_pct",
    color_continuous_scale="RdYlGn",
    text="appreciation_pct",
    title="Total Housing Price Appreciation 2010–2023 (%)",
    labels={"city": "City", "appreciation_pct": "Appreciation (%)"},
)
fig2.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig2.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    yaxis_title="Price Appreciation (%)",
    uniformtext_minsize=10,
)
fig2.show()


### Affordability index over time (all cities)

The most important chart. It adjusts price growth for income growth so you can see whether earnings kept up with the housing market. The dotted line at 1.0 is the baseline where incomes and prices moved together. Everything above it means prices won.

In [9]:
df_afford = pd.read_sql_query(
    "SELECT year, city, affordability_index FROM affordability ORDER BY city, year",
    conn,
)

fig3 = px.line(
    df_afford,
    x="year",
    y="affordability_index",
    color="city",
    markers=True,
    title="Housing Affordability Index — All Cities (higher = less affordable)",
    labels={"year": "Year", "affordability_index": "Affordability Index", "city": "City"},
)
fig3.update_traces(marker_size=4)
fig3.update_layout(
    hovermode="x unified",
    template="plotly_white",
    legend_title_text="City",
    yaxis_title="Affordability Index",
)
# Add a reference line at 1.0 (prices keeping pace with income growth)
fig3.add_hline(y=1.0, line_dash="dot", line_color="grey",
               annotation_text="Price = income growth baseline", annotation_position="right")
fig3.show()


### Biggest single-year price jump by city

The largest year-over-year price increase ever recorded for each city across the full 2000 to 2023 period. Shows which markets are most prone to sudden surges and how extreme those moments were.

In [10]:
df_peak = pd.read_sql_query(
    """
    SELECT
        city,
        ROUND(MAX(yoy_price_change), 1) AS peak_yoy_pct,
        year AS peak_year
    FROM affordability
    WHERE yoy_price_change = (
        SELECT MAX(yoy_price_change) FROM affordability a2 WHERE a2.city = affordability.city
    )
    GROUP BY city
    ORDER BY peak_yoy_pct DESC
    """,
    conn,
)

fig4 = px.bar(
    df_peak,
    x="city",
    y="peak_yoy_pct",
    color="peak_yoy_pct",
    color_continuous_scale="Reds",
    text="peak_yoy_pct",
    hover_data=["peak_year"],
    title="Peak Year-over-Year Price Increase by City (2000–2023)",
    labels={"city": "City", "peak_yoy_pct": "Peak YoY Increase (%)", "peak_year": "Year"},
)
fig4.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig4.update_layout(
    template="plotly_white",
    coloraxis_showscale=False,
    yaxis_title="Peak YoY Increase (%)",
    uniformtext_minsize=10,
)
fig4.show()


## 5. AI Insight

The key findings get passed to the Gemini API, which generates a plain-English market summary. The goal is what most stakeholders actually need: not raw numbers, but a clear narrative about what the data means. The API key is read from an environment variable and never stored in the notebook.

In [ ]:
# Pull the key findings we need for the prompt

# Total appreciation 2010–2023
df_appr_ai = pd.read_sql_query(
    """
    SELECT b.city,
           ROUND((l.price_index - b.price_index) * 100.0 / b.price_index, 1) AS pct
    FROM (SELECT city, price_index FROM housing_price_index WHERE year = 2010) b
    JOIN (SELECT city, price_index FROM housing_price_index WHERE year = 2023) l
        ON b.city = l.city
    ORDER BY pct DESC
    """,
    conn,
)

# Latest affordability index (2023)
df_afford_ai = pd.read_sql_query(
    """
    SELECT city, ROUND(affordability_index, 2) AS idx
    FROM affordability
    WHERE year = (SELECT MAX(year) FROM affordability)
    ORDER BY idx DESC
    """,
    conn,
)

# 2021 COVID-era YoY spike
df_covid_ai = pd.read_sql_query(
    "SELECT city, ROUND(yoy_price_change, 1) AS yoy FROM affordability WHERE year = 2021 ORDER BY yoy DESC",
    conn,
)

# Peak YoY per city
df_peak_ai = pd.read_sql_query(
    """
    SELECT city, ROUND(MAX(yoy_price_change), 1) AS peak, year AS peak_year
    FROM affordability
    WHERE yoy_price_change = (
        SELECT MAX(yoy_price_change) FROM affordability a2 WHERE a2.city = affordability.city
    )
    GROUP BY city ORDER BY peak DESC
    """,
    conn,
)

# Build the data summary for the prompt

lines = []
lines.append("TOTAL PRICE APPRECIATION 2010–2023:")
for _, r in df_appr_ai.iterrows():
    lines.append(f"  {r.city}: +{r.pct}%")

lines.append("\nLATEST AFFORDABILITY INDEX (2023, higher = less affordable):")
for _, r in df_afford_ai.iterrows():
    lines.append(f"  {r.city}: {r.idx}")

lines.append("\n2021 COVID-ERA YoY PRICE SPIKE:")
for _, r in df_covid_ai.iterrows():
    lines.append(f"  {r.city}: +{r.yoy}%")

lines.append("\nPEAK SINGLE-YEAR PRICE JUMP (any year 2000–2023):")
for _, r in df_peak_ai.iterrows():
    lines.append(f"  {r.city}: +{r.peak}% (in {int(r.peak_year)})")

data_block = "\n".join(lines)

prompt = f"""You are a Canadian real estate economist presenting to senior policy analysts.
Based on the following Statistics Canada data findings, write a 4–5 sentence plain-English
market summary. Be specific: reference city names, percentages, and years. Highlight the
divergence between high-cost coastal/Ontario markets and prairie cities, the COVID shock,
and what the affordability trajectory means for average households.

{data_block}
"""

# Call the Gemini API
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    display(Markdown("> ⚠️ **`GEMINI_API_KEY` not set** — set the environment variable and re-run this cell."))
else:
    client = genai.Client(api_key=api_key)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    summary = response.text
    display(Markdown(f"### AI Market Analysis\n\n{summary}"))


### AI Market Analysis

Canadian real estate markets have shown significant divergence and rapid appreciation from 2010 to 2023; while Edmonton prices rose just 10.0%, Ottawa saw a substantial 79.2% increase, and Winnipeg grew by 71.1%. By 2023, affordability in high-cost coastal and Ontario markets has severely deteriorated, with Ottawa's index at 138.49 and Toronto's at 103.23, contrasting sharply with Calgary (96.27) and Edmonton (82.91). The COVID-19 pandemic notably amplified this trend, acting as a significant shock: 2021 saw Ottawa's prices spike by 22.5% year-over-year and Montreal's by 17.0%, marking peak single-year jumps for those regions. This trajectory means average households, particularly in markets with high growth like Ottawa, Montreal, and Winnipeg, face escalating barriers to homeownership and long-term financial security.